# FabWatch：模擬設備ログの異常検知と実3D点検ロボット

このノートブックでは、模擬設備ログの生成、Isolation Forestによる異常検知、A*巡回計画、VTKによる実3D描画、日本語字幕付き動画とPDFレポートを作成する。

**注意：** 3D形状は実際の3Dレンダリングだが、ロボットは簡略化した差動二輪運動モデルで位置を更新する。剛体物理エンジンによる衝突・摩擦計算ではない。

## 1. Pythonパッケージと日本語フォントを導入

In [ ]:
%pip -q install "numpy>=1.24,<3" "pandas>=2,<3" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4" "opencv-python-headless>=4.8,<5" "reportlab>=4,<5" "joblib>=1.3,<2" "Pillow>=9.5,<13" "vtk>=9.3,<10" "nbformat>=5,<6"
!apt-get -qq update && apt-get install -y -qq fonts-noto-cjk ffmpeg >/dev/null

## 2. プロジェクト用ディレクトリの準備

In [ ]:
from pathlib import Path
import os
ROOT=Path('/content/FabWatch') if Path('/content').exists() else Path.cwd()/'FabWatch'
ROOT.mkdir(parents=True,exist_ok=True)
for folder in ('src','tests','assets','outputs','outputs/segments'):
    (ROOT/folder).mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
print('作業フォルダ:',ROOT)

## 3. ソースコード
以下のセルで作成するPythonファイルがGitHub公開用コードと一致する。

In [ ]:
%%writefile src/fabwatch.py
"""Anomaly scoring of simulated equipment logs and 2D inspection routes.

Isolation Forest sets the inspection order. A* plans each 2D route.
"""
from __future__ import annotations
import json, sqlite3, heapq, math
from dataclasses import dataclass
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, roc_auc_score
import joblib

SEED=20261026
MACHINES=[
    ('EQ01',(1.9,4.75),(1.9,5.73)),('EQ02',(4.0,4.75),(4.0,5.73)),
    ('EQ03',(6.35,4.75),(6.35,5.73)),('EQ04',(8.55,4.75),(8.55,5.73)),
    ('EQ05',(10.65,4.75),(10.65,5.73)),('EQ06',(10.65,2.24),(10.65,1.24)),
    ('EQ07',(7.25,2.24),(7.25,1.24)),('EQ08',(3.7,2.24),(3.7,1.24))]
MACHINE_IDS=[x[0] for x in MACHINES]
POINTS={'DOCK':(0.9,3.49), **{m:p for m,p,_ in MACHINES}}
BOXES=[(m,(cx-.67,cy-.42,cx+.67,cy+.42)) for m,_,(cx,cy) in MACHINES]
# Columns that robots cannot pass through.
BOXES+= [('COLUMN1',(5.3,3.06,5.8,3.95)),('COLUMN2',(9.32,3.06,9.82,3.95))]
GRID=.18; BOUNDS=(.28,11.85,.35,6.65); ROBOT_RADIUS=.18
FEATURES=['temp_C','vibration_mm_s','power_kW']


def _baseline(machine_i,t,rng):
    # Machine offsets + healthy operational fluctuation (all values synthetic).
    phase=2*np.pi*t/48.0
    return np.column_stack([
        38+1.7*(machine_i%3)+1.25*np.sin(phase)+rng.normal(0,.75,len(t)),
        1.25+.10*(machine_i%4)+.13*np.cos(phase)+rng.normal(0,.12,len(t)),
        6.7+.3*(machine_i%5)+.35*np.sin(phase+.8)+rng.normal(0,.24,len(t))])


def _dataset(seed:int, n:int=32, faulted:tuple=(), tag='eval'):
    rng=np.random.default_rng(seed)
    rows=[]
    for i,m in enumerate(MACHINE_IDS):
        t=np.arange(n)
        vals=_baseline(i,t,rng)
        if m in faulted:
            # Sensor drift sustained over the final 60% of this window.
            drift=np.maximum(0,(t-.4*n)/(.6*n))
            vals[:,0]+=6.0*drift
            vals[:,1]+=1.65*drift
            vals[:,2]+=1.2*drift
        for j in range(n):
            rows.append({'scenario':tag,'machine':m,'tick':j,'temp_C':float(vals[j,0]),
                'vibration_mm_s':float(vals[j,1]),'power_kW':float(vals[j,2]),
                'injected_fault':int(m in faulted)})
    return pd.DataFrame(rows)


def build_model():
    # Entire training AND calibration period is healthy. No faults are used to tune.
    train=_dataset(SEED,160,tag='healthy_train')
    normal_cal=pd.concat([_dataset(SEED+100+i,32,tag=f'cal_{i:02d}') for i in range(40)],ignore_index=True)
    scaler=StandardScaler().fit(train[FEATURES])
    model=IsolationForest(n_estimators=120, max_samples=256, contamination='auto',random_state=2026,n_jobs=1)
    model.fit(scaler.transform(train[FEATURES]))
    cal=pd.concat([window_scores(g,model,scaler) for _,g in normal_cal.groupby('scenario')],ignore_index=True)
    # This sets an operational baseline threshold, NOT an optimal F1-based threshold.
    threshold=float(np.quantile(cal['score'],.95))
    return model,scaler,threshold,train,normal_cal


def window_scores(df,model,scaler):
    d=df.copy()
    d['anomaly_score']=-model.score_samples(scaler.transform(d[FEATURES]))
    # The highest-scoring quartile highlights sustained anomalous readings, not one spike.
    result=d.groupby('machine',sort=False).agg(
       score=('anomaly_score',lambda x:float(np.mean(np.sort(np.asarray(x))[-max(1,len(x)//4):]))),
       injected_fault=('injected_fault','max'),
       mean_temp=('temp_C','mean'), mean_vibration=('vibration_mm_s','mean'),
       mean_power=('power_kW','mean')).reset_index()
    return result


def eval_scenarios(model,scaler,threshold,n_scenarios=60):
    rows=[]
    for s in range(n_scenarios):
        r=np.random.default_rng(4000+s)
        n_fault=int(r.integers(1,4))
        faulted=tuple(r.choice(MACHINE_IDS,n_fault,replace=False).tolist())
        test=_dataset(12000+s,32,faulted,tag=f'eval_{s:02d}')
        scored=window_scores(test,model,scaler)
        for q in scored.itertuples():
            rows.append(dict(scenario=s,machine=q.machine,injected_fault=int(q.injected_fault),
                             anomaly_score=float(q.score),predicted=int(q.score>threshold)))
    result=pd.DataFrame(rows)
    y=result.injected_fault.values; p=result.predicted.values
    precision,recall,f1,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    metrics={'n_scenarios':n_scenarios,'n_machine_windows':len(result),
      'precision':float(precision),'recall':float(recall),'f1':float(f1),
      'roc_auc':float(roc_auc_score(y,result.anomaly_score)),
      'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn)}
    return result,metrics


def collides(x,y,margin=ROBOT_RADIUS+.05):
    if not (BOUNDS[0]+margin<=x<=BOUNDS[1]-margin and BOUNDS[2]+margin<=y<=BOUNDS[3]-margin):return True
    for _,(x1,y1,x2,y2) in BOXES:
        if x1-margin<=x<=x2+margin and y1-margin<=y<=y2+margin:return True
    return False


def nearest_node(p):return (int(round((p[0]-BOUNDS[0])/GRID)),int(round((p[1]-BOUNDS[2])/GRID)))
def node_xy(n):return (BOUNDS[0]+n[0]*GRID,BOUNDS[2]+n[1]*GRID)

@lru_cache(maxsize=128)
def plan_path(start_id:str,end_id:str):
    a=POINTS[start_id];b=POINTS[end_id]
    start=nearest_node(a);end=nearest_node(b)
    if start==end:return (a,b)
    pq=[(0.,start)];cost={start:0.};prev={};visited=set()
    while pq:
        _,node=heapq.heappop(pq)
        if node in visited:continue
        visited.add(node)
        if node==end:break
        for dx,dy in [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]:
            next_node=(node[0]+dx,node[1]+dy)
            xx,yy=node_xy(next_node)
            if collides(xx,yy):continue
            if dx*dy and (collides(*node_xy((node[0]+dx,node[1]))) or collides(*node_xy((node[0],node[1]+dy)))):continue
            nc=cost[node]+math.hypot(dx,dy)*GRID
            if nc<cost.get(next_node,float('inf')):
                cost[next_node]=nc;prev[next_node]=node
                heapq.heappush(pq,(nc+GRID*math.hypot(end[0]-next_node[0],end[1]-next_node[1]),next_node))
    if end not in cost:raise ValueError(f'No route {start_id} to {end_id}')
    path=[end];cur=end
    while cur!=start:
        cur=prev[cur];path.append(cur)
    path.reverse()
    pts=[a]+[node_xy(n) for n in path[1:-1]]+[b]
    # Reduce zigzag while retaining collision checks with dense points.
    simplified=[pts[0]];i=0
    while i<len(pts)-1:
        j=len(pts)-1
        while j>i+1:
            if segment_clear(pts[i],pts[j]):break
            j-=1
        simplified.append(pts[j]);i=j
    return tuple(simplified)


def segment_clear(a,b):
    dis=math.dist(a,b);n=max(1,int(math.ceil(dis/.07)))
    for k in range(n+1):
        t=k/n
        if collides(a[0]+t*(b[0]-a[0]),a[1]+t*(b[1]-a[1])):return False
    return True


def route_length(points):return sum(math.dist(a,b) for a,b in zip(points[:-1],points[1:]))


def build_route(order:list[str],speed=.74,inspection_s=6.0):
    segments=[];total_m=0.;t=0.;here='DOCK';first_fault_t=None
    for machine in order:
        p=plan_path(here,machine);dist=route_length(p)
        segments.append({'from':here,'to':machine,'polyline':p,'start_s':t,'end_s':t+dist/speed,'inspection_end_s':t+dist/speed+inspection_s,'distance_m':dist})
        t+=dist/speed+inspection_s;total_m+=dist;here=machine
    return segments,total_m,t


def first_fault_time(order,segments,faults):
    for machine,seg in zip(order,segments):
        if machine in faults:return float(seg['inspection_end_s'])
    return None


def run_all(out='outputs', n_scenarios=60):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    model,scaler,threshold,train,cal=build_model()
    records,metrics=eval_scenarios(model,scaler,threshold,n_scenarios=n_scenarios)
    train.to_csv(out/'healthy_training.csv',index=False)
    records.to_csv(out/'heldout_evaluation.csv',index=False)
    # Demo scenario is excluded from the aggregate evaluation; the anomaly labels are synthetic.
    demo=_dataset(118001,36,('EQ06','EQ07'),tag='demo')
    results=window_scores(demo,model,scaler)
    results['predicted_anomaly']=(results['score']>threshold).astype(int)
    demo.to_csv(out/'demo_equipment_logs.csv',index=False)
    results.to_csv(out/'demo_ranking.csv',index=False)
    with sqlite3.connect(out/'equipment_logs.sqlite') as conn:
        demo.to_sql('sensor_logs',conn,index=False,if_exists='replace')
        results.to_sql('anomaly_ranking',conn,index=False,if_exists='replace')
        conn.execute('CREATE INDEX IF NOT EXISTS idx_machine_time ON sensor_logs(machine,tick)')
    # No target-label access at scheduling time. AI order uses only model scores.
    fixed=MACHINE_IDS[:]
    ai=results.sort_values('score',ascending=False)['machine'].tolist()
    routes={}
    for name,order in [('fixed',fixed),('ai',ai)]:
        segments,d,t=build_route(order)
        routes[name]={'order':order,'segments':segments,'distance_m':round(d,3),
            'duration_s':round(t,3), 'first_fault_inspected_s':round(first_fault_time(order,segments,{'EQ06','EQ07'}),3)}
    # Compare schedules over held-out, independently seeded synthetic scenarios.
    # The true fault labels are used only to score the routes after scheduling.
    simrows=[]
    for sid,group in records.groupby('scenario',sort=True):
        hazards=set(group.loc[group.injected_fault==1,'machine'])
        ai_order=group.sort_values('anomaly_score',ascending=False)['machine'].tolist()
        ss,sd,st=build_route(ai_order)
        fs,fd,ft=build_route(fixed)
        simrows.append({'scenario':int(sid),'fault_count':len(hazards),
           'fixed_first_fault_s':first_fault_time(fixed,fs,hazards),
           'ai_first_fault_s':first_fault_time(ai_order,ss,hazards),
           'fixed_route_m':fd,'ai_route_m':sd,
           'fixed_total_s':ft,'ai_total_s':st})
    simdf=pd.DataFrame(simrows)
    simdf.to_csv(out/'schedule_comparison.csv',index=False)
    diff=simdf.fixed_first_fault_s-simdf.ai_first_fault_s
    metrics['eval_avg_fixed_first_fault_s']=float(simdf.fixed_first_fault_s.mean())
    metrics['eval_avg_ai_first_fault_s']=float(simdf.ai_first_fault_s.mean())
    metrics['eval_mean_first_fault_gain_s']=float(diff.mean())
    metrics['eval_first_fault_improved_n']=int((diff>1e-6).sum())
    metrics['eval_first_fault_worse_n']=int((diff<-1e-6).sum())
    metrics['eval_avg_fixed_route_m']=float(simdf.fixed_route_m.mean())
    metrics['eval_avg_ai_route_m']=float(simdf.ai_route_m.mean())
    rg=np.random.default_rng(5561)
    idx=rg.integers(0,len(diff),size=(4000,len(diff)))
    metrics['paired_bootstrap_gain_ci95_s']=[float(x) for x in np.percentile(diff.to_numpy()[idx].mean(axis=1),[2.5,97.5])]
    metrics['threshold']=threshold
    metrics['demo_faults']=['EQ06','EQ07']
    metrics['demo_fixed_first_fault_s']=routes['fixed']['first_fault_inspected_s']
    metrics['demo_ai_first_fault_s']=routes['ai']['first_fault_inspected_s']
    metrics['demo_fixed_distance_m']=routes['fixed']['distance_m']
    metrics['demo_ai_distance_m']=routes['ai']['distance_m']
    metrics['demo_fixed_duration_s']=routes['fixed']['duration_s']
    metrics['demo_ai_duration_s']=routes['ai']['duration_s']
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding='utf-8')
    (out/'routes.json').write_text(json.dumps(routes,indent=2,ensure_ascii=False),encoding='utf-8')
    joblib.dump({'model':model,'scaler':scaler,'threshold':threshold,'features':FEATURES},out/'isolation_forest.joblib')
    return metrics

if __name__=='__main__':
    m=run_all()
    print(json.dumps(m,indent=2,ensure_ascii=False))


In [ ]:
%%writefile src/robot_motion.py
"""Differential-drive kinematic tracking and 2D range ray calculations.
Used only for pose generation; 3D visualization is rendered separately via VTK.
"""
import math
from fabwatch import BOXES,POINTS,BOUNDS
WHEEL_R=0.06
AXLE=0.28
ROBOT_L=.42
ROBOT_W=.28
ROBOT_H=.14
MAX_SENSOR=1.35

def clamp(x,a,b):return max(a,min(b,x))

def ray_box_intersection(origin, theta, rect, max_range=MAX_SENSOR):
    x0,y0=origin
    x1,y1,x2,y2=rect
    step=0.03; s=0.0
    while s<=max_range:
        x=x0+math.cos(theta)*s; y=y0+math.sin(theta)*s
        if x1<=x<=x2 and y1<=y<=y2:
            return s
        s+=step
    s=0.0
    while s<=max_range:
        x=x0+math.cos(theta)*s; y=y0+math.sin(theta)*s
        if x<BOUNDS[0] or x>BOUNDS[1] or y<BOUNDS[2] or y>BOUNDS[3]:
            return s
        s+=step
    return max_range


def sensor_distances(x,y,yaw):
    sensors={'s_front':MAX_SENSOR,'s_left':MAX_SENSOR,'s_right':MAX_SENSOR}
    rays={'s_front':yaw,'s_left':yaw+math.radians(35),'s_right':yaw-math.radians(35)}
    for key,ang in rays.items():
        best=MAX_SENSOR
        for _,rect in BOXES:
            best=min(best, ray_box_intersection((x,y),ang,rect,MAX_SENSOR))
        sensors[key]=best
    return sensors


def simulate_route(route,speed=0.74,inspection_s=6.0,dt=0.1):
    x,y=POINTS['DOCK']; yaw=0.0; t=0.0; visited=[]; states=[]
    v_nom=speed
    for seg in route['segments']:
        polyline=[tuple(p) for p in seg['polyline']]
        target_machine=seg['to']
        waypoint_idx=1
        while waypoint_idx < len(polyline):
            tx,ty=polyline[waypoint_idx]
            if math.dist((x,y),(tx,ty))<0.12 and waypoint_idx < len(polyline)-1:
                waypoint_idx += 1; tx,ty=polyline[waypoint_idx]
            dx=tx-x; dy=ty-y
            desired=math.atan2(dy,dx)
            err=(desired-yaw+math.pi)%(2*math.pi)-math.pi
            v=v_nom*clamp(1-0.35*abs(err),0.22,1.0)
            if math.dist((x,y),(tx,ty))<0.18 and waypoint_idx==len(polyline)-1:
                v=min(v,0.28)
            omega=clamp(2.6*err,-1.8,1.8)
            vl=v-omega*AXLE/2; vr=v+omega*AXLE/2
            v_robot=(vr+vl)/2; w_robot=(vr-vl)/AXLE
            x += v_robot*math.cos(yaw)*dt; y += v_robot*math.sin(yaw)*dt; yaw += w_robot*dt
            sens=sensor_distances(x,y,yaw)
            states.append({'t':t,'x':x,'y':y,'yaw':yaw,'current':target_machine,'visited':visited.copy(),
                           'mode':'move','vl':vl/WHEEL_R,'vr':vr/WHEEL_R,**sens})
            t+=dt
            if waypoint_idx==len(polyline)-1 and math.dist((x,y),polyline[-1])<0.08:
                break
        x,y=polyline[-1]
        equip=[r for r in BOXES if r[0]==target_machine][0][1]
        cx=(equip[0]+equip[2])/2; cy=(equip[1]+equip[3])/2; yaw=math.atan2(cy-y,cx-x)
        steps=max(1,int(round(inspection_s/dt)))
        for k in range(steps):
            sens=sensor_distances(x,y,yaw)
            states.append({'t':t,'x':x,'y':y,'yaw':yaw,'current':target_machine,'visited':visited.copy(),
                           'mode':'inspect','inspect_progress':(k+1)/steps,'vl':0.0,'vr':0.0,**sens})
            t+=dt
        visited=visited+[target_machine]
    for _ in range(int(2/dt)):
        sens=sensor_distances(x,y,yaw)
        states.append({'t':t,'x':x,'y':y,'yaw':yaw,'current':None,'visited':visited.copy(),'mode':'done','vl':0.0,'vr':0.0,**sens})
        t+=dt
    return states


def get_state(trace, elapsed, playback_duration):
    total_t=trace[-1]['t'] if trace else 1.0
    sim_t=clamp(elapsed/playback_duration,0.0,1.0)*total_t
    idx=min(len(trace)-1, int(sim_t/0.1))
    return trace[idx], sim_t, total_t




In [ ]:
%%writefile src/scene3d.py
"""Actual 3D geometry and perspective camera rendered with VTK.

3D visual simulation: wheel rotation and chassis pose follow differential-drive
kinematics, not an articulated rigid-body physics solver. Distance readings are
3D horizontal rays intersecting equipment bounding boxes.
"""
from __future__ import annotations
import math
import numpy as np
import vtk
from vtk.util.numpy_support import vtk_to_numpy
from fabwatch import BOXES, BOUNDS


def rgb(r,g,b): return tuple(float(v)/255 for v in (r,g,b))

class Fab3D:
    def __init__(self,width=800,height=450):
        self.width,self.height=width,height
        self.ren=vtk.vtkRenderer()
        self.ren.SetBackground(0.064,0.096,0.13)
        self.ren.GradientBackgroundOn()
        self.ren.SetBackground2(0.22,0.29,0.32)
        self.win=vtk.vtkRenderWindow()
        self.win.SetOffScreenRendering(1)
        self.win.SetSize(width,height)
        self.win.AddRenderer(self.ren)
        self.ren.SetUseFXAA(False)
        self._build_plant()
        self._build_robot()
        self._set_camera()
        self.window_filter=vtk.vtkWindowToImageFilter()
        self.window_filter.SetInput(self.win)
        self.window_filter.SetInputBufferTypeToRGB()
        self.ren.ResetCameraClippingRange()
        self.win.Render()

    def mat(self,actor,c,ambient=0.25,diffuse=0.72,specular=0.18):
        p=actor.GetProperty();p.SetColor(*rgb(*c));p.SetAmbient(ambient);p.SetDiffuse(diffuse)
        p.SetSpecular(specular);p.SetSpecularPower(23)
        return actor
    def box(self,pos,dims,c):
        s=vtk.vtkCubeSource();s.SetXLength(dims[0]);s.SetYLength(dims[1]);s.SetZLength(dims[2])
        m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort())
        a=vtk.vtkActor();a.SetMapper(m);a.SetPosition(*pos);self.mat(a,c);self.ren.AddActor(a)
        return a
    def cylinder(self,pos,radius,height,c):
        s=vtk.vtkCylinderSource();s.SetRadius(radius);s.SetHeight(height);s.SetResolution(18)
        m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort())
        a=vtk.vtkActor();a.SetMapper(m);a.SetPosition(*pos);self.mat(a,c);self.ren.AddActor(a)
        return a
    def sphere(self,pos,r,c):
        s=vtk.vtkSphereSource();s.SetRadius(r);s.SetThetaResolution(16);s.SetPhiResolution(12)
        m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort())
        a=vtk.vtkActor();a.SetMapper(m);a.SetPosition(*pos);self.mat(a,c);self.ren.AddActor(a);return a
    def label(self,text,pos,scale=0.14,c=(245,247,247)):
        s=vtk.vtkVectorText();s.SetText(text)
        m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort())
        a=vtk.vtkFollower();a.SetMapper(m);a.SetScale(scale,scale,scale);a.SetPosition(*pos)
        self.mat(a,c,ambient=0.7,diffuse=0.3,specular=0)
        self.ren.AddActor(a);a.SetCamera(self.ren.GetActiveCamera());return a
    def line_actor(self,points,c,width=2):
        poly=vtk.vtkPolyData();pts=vtk.vtkPoints()
        for v in points:pts.InsertNextPoint(*v)
        poly.SetPoints(pts);lines=vtk.vtkCellArray();lines.InsertNextCell(len(points))
        for i in range(len(points)):lines.InsertCellPoint(i)
        poly.SetLines(lines)
        m=vtk.vtkPolyDataMapper();m.SetInputData(poly)
        a=vtk.vtkActor();a.SetMapper(m);a.GetProperty().SetColor(*rgb(*c));a.GetProperty().SetLineWidth(width);a.GetProperty().SetAmbient(1)
        self.ren.AddActor(a);return a
    def _build_plant(self):
        # 3D cleanroom production floor, marked inspection aisle, structural elements.
        self.box((6.05,3.5,-.13),(12.2,7.1,.25),(176,189,190))
        self.box((6.05,3.47,.008),(11.9,.82,.015),(87,117,124))
        for y in (2.99,3.92):
            self.box((6.05,y,.024),(11.5,.045,.02),(246,187,62))
        for x in np.arange(.7,11.7,1.05):
            self.box((x,3.48,.022),(.48,.018,.018),(218,230,227))
        for ix in (1,2,3,4,5,6,7,8,9,10,11):
            self.line_actor([(ix,0.25,.02),(ix,6.75,.02)],(143,160,164),1)
        for yy in (1,2,3,4,5,6):
            self.line_actor([(.25,yy,.021),(11.85,yy,.021)],(143,160,164),1)
        # perimeter structural framing, ventilation above equipment
        self.box((6.05,6.78,1.1),(12.2,.10,2.2),(171,183,188))
        for x in (0.35,11.77):
            self.box((x,.25,1.35),(.10,.18,2.7),(114,129,133))

        # equipment cabinets using original collision rectangles exactly
        self.lamps={}
        self.cabinets={}
        for eq,(x1,y1,x2,y2) in BOXES:
            x,y=(x1+x2)/2,(y1+y2)/2
            if eq.startswith('COLUMN'):
                self.box((x,y,.88),(x2-x1,y2-y1,1.76),(115,132,137));continue
            w,d=x2-x1,y2-y1
            c=(205,218,220) if eq not in ('EQ06','EQ07') else (196,210,217)
            self.cabinets[eq]=self.box((x,y,.68),(w,d,1.36),c)
            self.box((x,y2+.012,.73),(w*.72,.025,.76),(64,86,99))
            for z in (.46,.59,.72,.85):self.box((x-.36,y2+.035,z),(.36,.012,.018),(115,159,169))
            self.box((x+.36,y2+.047,.93),(.16,.03,.18),(36,64,71))
            lamp=self.sphere((x2-.15,y1+.15,1.43),.085,(46,193,120))
            self.lamps[eq]=lamp
            self.label(eq,(x-.31,y2+.045,1.21),.18)
            # service cabinet front panel handle
            self.box((x+.42,y2+.048,.7),(.025,.035,.18),(210,221,231))
        # service dock and floor markings
        self.box((.86,3.5,.075),(.86,.65,.15),(70,128,141))
        self.label('DOCK',(.52,3.08,.14),.11)
        for x in (0.5,1.2):
            self.box((x,3.14,.036),(.12,.055,.03),(239,184,50))
        # hazard striped stops, static props not changing navigable footprint
        for x in (5.55,9.57):
            self.box((x,3.16,.04),(.36,.08,.04),(220,164,65))
        light=vtk.vtkLight();light.SetPosition(3,-3,11);light.SetFocalPoint(6,3,0);light.SetIntensity(0.8);self.ren.AddLight(light)
        light2=vtk.vtkLight();light2.SetPosition(14,8,10);light2.SetFocalPoint(4,3,0);light2.SetIntensity(.48);self.ren.AddLight(light2)
    def _build_robot(self):
        self.assembly=vtk.vtkAssembly();self.ren.AddActor(self.assembly)
        def child_box(pos,dims,c):
            s=vtk.vtkCubeSource();s.SetXLength(dims[0]);s.SetYLength(dims[1]);s.SetZLength(dims[2]);m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort());a=vtk.vtkActor();a.SetMapper(m);a.SetPosition(*pos);self.mat(a,c);self.assembly.AddPart(a);return a
        def child_cyl(pos,r,h,c):
            s=vtk.vtkCylinderSource();s.SetRadius(r);s.SetHeight(h);s.SetResolution(24);m=vtk.vtkPolyDataMapper();m.SetInputConnection(s.GetOutputPort());a=vtk.vtkActor();a.SetMapper(m);a.SetPosition(*pos);self.mat(a,c);self.assembly.AddPart(a);return a
        child_box((0,0,.24),(.50,.32,.22),(35,118,149))
        child_box((.10,0,.38),(.23,.23,.06),(51,76,87))
        child_box((.21,0,.24),(.07,.23,.055),(248,197,73))
        self.wheels=[]
        for side in (-1,1):
            a=child_cyl((0,side*.195,.12),.115,.085,(30,41,47))
            self.wheels.append(a)
        child_cyl((-.19,0,.075),.065,.12,(90,98,103))
        child_box((.055,0,.49),(.09,.09,.12),(97,139,150))
        child_box((.055,0,.57),(.20,.15,.035),(32,66,83))
        self.beams=[]
        for c in ((65,229,220),(252,188,72),(114,224,138)):
            l=vtk.vtkLineSource();m=vtk.vtkPolyDataMapper();m.SetInputConnection(l.GetOutputPort());a=vtk.vtkActor();a.SetMapper(m);a.GetProperty().SetColor(*rgb(*c));a.GetProperty().SetLineWidth(3);a.GetProperty().SetAmbient(1);self.ren.AddActor(a);self.beams.append((l,a))
        self.ring=self.cylinder((0,0,.015),.31,.014,(30,169,195))
        self.ring.VisibilityOff()
        self._wheel_spin=[0.,0.];self._last_sim_t=None
    def _set_camera(self):
        cam=self.ren.GetActiveCamera()
        cam.SetPosition(13.3,-10.0,10.2)
        cam.SetFocalPoint(6.05,3.4,.35)
        cam.SetViewUp(0,0,1)
        cam.SetViewAngle(35)
        self.ren.ResetCameraClippingRange()
    def set_state(self,s,pulse=0):
        x,y,yaw=s['x'],s['y'],s['yaw']
        self.assembly.SetPosition(x,y,0)
        self.assembly.SetOrientation(0,0,math.degrees(yaw))
        t=s.get('t',0.)
        if self._last_sim_t is None:dt=0
        else:dt=max(0,min(.6,t-self._last_sim_t))
        self._last_sim_t=t
        for i,vel in enumerate((s['vl'],s['vr'])):
            self._wheel_spin[i]=(self._wheel_spin[i]+math.degrees(vel*dt))%360
            self.wheels[i].SetOrientation(0,self._wheel_spin[i],0)
        # sensor beams in world XYZ, intersect cabinet volumes at sensor plane z=.58
        z=.58
        for ((src,actor),off) in zip(self.beams,(0,math.radians(36),-math.radians(36))):
            angle=yaw+off;dist=self.sensor_3d(x,y,angle,z)
            src.SetPoint1(x+.06*math.cos(yaw),y+.06*math.sin(yaw),z)
            src.SetPoint2(x+dist*math.cos(angle),y+dist*math.sin(angle),z)
            src.Modified()
        self.ring.SetPosition(x,y,.035)
        self.ring.SetScale(1+.18*math.sin(pulse),1+.18*math.sin(pulse),1)
        if s['mode']=='inspect':self.ring.VisibilityOn()
        else:self.ring.VisibilityOff()
        for eq,lamp in self.lamps.items():
            if eq==s.get('current') and s['mode']=='inspect':
                lamp.GetProperty().SetColor(*rgb(243,198,66))
            else:
                lamp.GetProperty().SetColor(*rgb(*((234,79,82) if eq in ('EQ06','EQ07') else (46,193,120))))
    @staticmethod
    def sensor_3d(x,y,angle,z,max_range=1.50):
        # 3D ray originates above floor; cabinets and columns have known 3D extents.
        origin=np.array([x,y,z]);dx,dy=math.cos(angle),math.sin(angle)
        smallest=max_range
        for _,(x1,y1,x2,y2) in BOXES:
            low=0.; high=max_range
            for o,d,a,b in ((x,dx,x1,x2),(y,dy,y1,y2)):
                if abs(d)<1e-8:
                    if o<a or o>b:high=-1;break
                else:
                    t1=(a-o)/d;t2=(b-o)/d
                    low=max(low,min(t1,t2));high=min(high,max(t1,t2))
            if high>=max(low,0):smallest=min(smallest,max(low,0))
        return smallest
    def frame(self):
        self.win.Render();self.window_filter.Modified();self.window_filter.Update()
        image=self.window_filter.GetOutput();w,h,_=image.GetDimensions()
        data=vtk_to_numpy(image.GetPointData().GetScalars()).reshape(h,w,3)
        return np.ascontiguousarray(data[::-1,:,::-1])


In [ ]:
%%writefile src/make_video.py
"""Render a true 3D VTK factory scene and Japanese 2D informational overlays.

The model executes 2D differential-drive kinematics and poses 3D meshes.
It is not a rigid-body physics simulation. AI selects equipment order only.
"""
from __future__ import annotations
import json, math, os, subprocess
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image,ImageDraw,ImageFont
from fabwatch import MACHINE_IDS,POINTS
from robot_motion import simulate_route,get_state
from scene3d import Fab3D

W,H,FPS=1280,720,10
SCENE_W,SCENE_H=860,490
JP_REG='/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
JP_BOLD='/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc'
DARK=(15,28,39); PANEL=(24,42,54); CYAN=(83,218,219); ORANGE=(252,192,95)
RED=(241,119,111); GREEN=(107,216,163); WHITE=(235,245,246); MUTED=(161,180,192)

def jp_font(size,bold=False):return ImageFont.truetype(JP_BOLD if bold else JP_REG,size)

def panel(draw,rect,fill=PANEL,outline=(58,84,96),radius=10):
    draw.rounded_rectangle(rect,radius=radius,fill=fill,outline=outline,width=2)

def txt(d,x,y,s,size=20,color=WHITE,bold=False):
    d.text((int(x),int(y)),str(s),font=jp_font(size,bold),fill=color)

def base(title,small):
    im=Image.new('RGB',(W,H),DARK)
    d=ImageDraw.Draw(im)
    d.rectangle((0,0,W,80),fill=(12,25,36))
    txt(d,28,12,'FABWATCH',34,bold=True)
    txt(d,278,20,title,23,CYAN,bold=True)
    txt(d,1040,28,small,15,MUTED)
    d.line((28,77,1250,77),fill=CYAN,width=2)
    return im,d

def main_3d(im,scene_img):
    x,y=25,95
    rgb=cv2.cvtColor(scene_img,cv2.COLOR_BGR2RGB)
    im.paste(Image.fromarray(rgb).resize((SCENE_W,SCENE_H)),(x,y))
    d=ImageDraw.Draw(im);d.rectangle((x,y,x+SCENE_W,y+SCENE_H),outline=(99,143,155),width=2)
    return d

def status_box(d,state,rank,mode=''):
    panel(d,(899,96,1252,585))
    txt(d,923,108,'設備の異常スコア',22,bold=True)
    rows=rank.sort_values('score',ascending=False)
    for i,r in enumerate(rows.itertuples()):
        yy=151+i*37
        current=r.machine==state.get('current')
        if current:d.rounded_rectangle((910,yy-5,1242,yy+31),fill=(48,73,82),radius=4)
        txt(d,922,yy,r.machine,19,WHITE,current)
        xx=994; bw=155
        d.rectangle((xx,yy+11,xx+bw,yy+23),fill=(47,65,72))
        v=max(.02,min(1,(float(r.score)-.42)/.29))
        cc=RED if r.predicted_anomaly else CYAN
        d.rectangle((xx,yy+11,xx+int(bw*v),yy+23),fill=cc)
        txt(d,1164,yy,f'{r.score:.3f}',16,cc)
    d.line((913,466,1238,466),fill=(74,96,107),width=1)
    txt(d,916,475,'ロボットの動作',19,WHITE,True)
    action='点検停止中' if state['mode']=='inspect' else '移動中'
    txt(d,916,505,f'状態：{action}',16)
    txt(d,916,534,f'左右モータ：{state["vl"]:.1f} / {state["vr"]:.1f} rad/s',14,MUTED)
    txt(d,916,559,f'距離（前方）：{state["s_front"]:.2f} m',15,ORANGE)

def footer(im,main,sub):
    d=ImageDraw.Draw(im)
    panel(d,(25,604,1252,696),fill=(21,37,49),radius=7)
    txt(d,46,614,main,26,WHITE,True)
    txt(d,46,652,sub,18,MUTED)
    return im

def scene_camera(s3d,state,mode):
    cam=s3d.ren.GetActiveCamera()
    if mode in ('fixed','ai'):
        x,y=state['x'],state['y']
        positive=state.get('current') in ('EQ06','EQ07','EQ08')
        yy=y+(4.5 if positive else -4.5)
        cam.SetPosition(x+2.9,yy,3.6)
        cam.SetFocalPoint(x,y,.32)
        cam.SetViewUp(0,0,1)
        cam.SetViewAngle(44)
    else:
        cam.SetPosition(13.3,-10.0,10.2)
        cam.SetFocalPoint(6.05,3.4,.35)
        cam.SetViewUp(0,0,1)
        cam.SetViewAngle(35)
    s3d.ren.ResetCameraClippingRange()

def highlight_points(im,state):
    if state['mode']!='inspect':return
    d=ImageDraw.Draw(im)
    d.rounded_rectangle((45,504,410,575),fill=(20,69,63),outline=CYAN,width=2,radius=7)
    txt(d,62,515,f'点検中：{state["current"]}',28,WHITE,True)
    p=float(state.get('inspect_progress',0))
    d.rectangle((62,555,376,567),fill=(54,89,91))
    d.rectangle((62,555,62+int(314*p),567),fill=CYAN)

def scene_frame(section,elapsed,risk,met,routes,demo,traces,s3d):
    if section in ('fixed','ai'):
        screen_seconds=48 if section=='fixed' else 56
        state,simt,total_t=get_state(traces[section],elapsed,screen_seconds)
        heading='固定順巡回' if section=='fixed' else 'AI異常度優先巡回'
        playback=total_t/screen_seconds
    else:
        state=traces['ai'][0]
        heading='仮想設備点検エリア'
        simt=0;playback=1
    s3d.set_state(state,elapsed)
    scene_camera(s3d,state,section)
    image,d=base(heading,'3Dレンダリング：VTK')
    d=main_3d(image,s3d.frame())
    status_box(d,state,risk,section)
    if section in ('fixed','ai'):
        highlight_points(image,state)
        first=routes[section]['first_fault_inspected_s']
        dist=routes[section]['distance_m']
        primary='固定方式：設備番号順に巡回' if section=='fixed' else 'AI方式：異常スコアの高い設備から巡回'
        secondary=f'再生約{playback:.2f}倍   |   初回異常点検 {first:.1f}秒   |   計画移動距離 {dist:.1f}m'
        footer(image,primary,secondary)
    else:
        footer(image,'異常検知結果から点検先を決定','8設備の模擬ログを分析し、AIのスコア順で点検ルートを変更')
    return cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2BGR)

def ds_frame(section,elapsed,rank,met,out):
    headings={'train':'学習・閾値設定','validate':'未学習データでの評価','results':'点検順の比較結果','limitations':'適用範囲と改善点'}
    image,d=base(headings[section],'FABWATCH / DATA ANALYSIS')
    panel(d,(24,97,1253,584))
    if section in ('train','validate'):
        pic='sensor_signals.png' if section=='train' else 'score_distribution.png'
        chart=Image.open(out/pic).convert('RGB')
        chart.thumbnail((835,420))
        image.paste(chart,(44,125))
        d=ImageDraw.Draw(image)
        panel(d,(905,115,1235,552),fill=(30,51,61))
        if section=='train':
            lines=[('01  正常ログを生成', '温度・振動・電力を使用'),('02  学習と閾値を分離','正常履歴でIsolation Forestを学習'),('03  別の正常40条件','スコアの95%点を閾値とする')]
            foot=('学習データと閾値設定用データを分離','異常ラベルを使って閾値を調整していない。')
        else:
            lines=[('評価条件：60セット','合計480設備窓、1～3台に異常注入'),('適合率 / Precision',f'{met["precision"]:.3f}'),('再現率 / Recall',f'{met["recall"]:.3f}'),('F1スコア',f'{met["f1"]:.3f}')]
            foot=('学習していない60条件で異常検知を評価','評価結果は人工データのドリフト検出性能であり実機の故障予測ではない。')
        for i,(a,b) in enumerate(lines):
            yy=139+i*(90 if section=='train' else 80)
            txt(d,919,yy,a,20,CYAN,True)
            txt(d,919,yy+32,b,16,WHITE)
        footer(image,*foot)
    elif section=='results':
        txt(d,55,117,'初回異常設備の点検完了までの平均時間',26,WHITE,True)
        barwidth=650
        for i,(label,sec,color) in enumerate([('固定方式',met['eval_avg_fixed_first_fault_s'],MUTED),('AI方式',met['eval_avg_ai_first_fault_s'],CYAN)]):
            y=205+i*108
            txt(d,62,y,label,24,WHITE,True)
            d.rectangle((240,y,240+barwidth,y+37),fill=(39,61,71))
            d.rectangle((240,y,240+int(barwidth*sec/35),y+37),fill=color)
            txt(d,940,y-3,f'{sec:.1f}秒',26,color,True)
        txt(d,75,449,f'AIが早かった条件：{met["eval_first_fault_improved_n"]} / 60',23,CYAN,True)
        txt(d,75,499,f'平均移動距離：固定 {met["eval_avg_fixed_route_m"]:.1f}m  /  AI {met["eval_avg_ai_route_m"]:.1f}m',22,ORANGE)
        footer(image,'初回点検は早まったが、移動距離は増えた','移動時間を考慮しないスコア順巡回には改善の余地がある。')
    else:
        content=[
          '・設備データ、工場配置、異常注入条件はいずれも人工的に設定した。',
          '・AIが決めるのは点検順序。モーター制御は差動二輪の簡略運動モデル。',
          '・映像はVTKの立体形状と透視投影を使う実際の3Dレンダリング。',
          '・剛体力学エンジンによる摩擦・衝突・荷重などの実機再現ではない。',
          '・今後は移動時間を含めた最適化と、実際の計測ログによる検証が必要。'
        ]
        for i,s in enumerate(content):txt(d,64,155+i*76,s,23,WHITE)
        footer(image,'FabWatch / 最終課題','異常検知・点検順序・ロボットの動作を分けて設計した。')
    return cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2BGR)

def make_main_video(root:Path,fps=FPS):
    out=root/'outputs';out.mkdir(parents=True,exist_ok=True)
    risk=pd.read_csv(out/'demo_ranking.csv');demo=pd.read_csv(out/'demo_equipment_logs.csv')
    metrics=json.loads((out/'metrics.json').read_text())
    routes=json.loads((out/'routes.json').read_text())
    traces={k:simulate_route(v) for k,v in routes.items()}
    s3d=Fab3D(SCENE_W,SCENE_H)
    # 6 FPS intentionally balances rendering cost and Colab CPU accessibility.
    scenes=[('intro',6),('train',13),('validate',11),('fixed',48),('ai',56),('results',13),('limitations',6)]
    main=out/'FabWatch_実3D_本編.mp4'
    writer=cv2.VideoWriter(str(main),cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
    if not writer.isOpened():raise RuntimeError('Cannot create MP4')
    for section,seconds in scenes:
        for n in range(int(seconds*fps)):
            if section in ('train','validate','results','limitations'):
                frame=ds_frame(section,n/fps,risk,metrics,out)
            else:
                frame=scene_frame(section,n/fps,risk,metrics,routes,demo,traces,s3d)
            writer.write(frame)
        print('rendered',section,flush=True)
    writer.release()
    return main

def add_op_ed(root:Path,main:Path,op:Path,ed:Path):
    """Use ffmpeg concat with fps normalization; preserves 3-second OP/ED durations."""
    output=root/'outputs'/'FabWatch_実3D_提出動画_OP_ED付き.mp4'
    cmd=['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(op),'-i',str(main),'-i',str(ed),
         '-filter_complex',f'[0:v]fps={FPS},scale={W}:{H},setsar=1[v0];[1:v]fps={FPS},scale={W}:{H},setsar=1[v1];[2:v]fps={FPS},scale={W}:{H},setsar=1[v2];[v0][v1][v2]concat=n=3:v=1:a=0[v]',
         '-map','[v]','-c:v','libx264','-preset','fast','-crf','22','-pix_fmt','yuv420p','-movflags','+faststart',str(output)]
    subprocess.run(cmd,check=True)
    return output

if __name__=='__main__':
    root=Path(__file__).resolve().parents[1]
    main=make_main_video(root)
    op,ed=root/'assets'/'OP.mp4',root/'assets'/'ED.mp4'
    if op.exists() and ed.exists():print('submission',add_op_ed(root,main,op,ed))
    else:print('main only',main)


In [ ]:
%%writefile src/render_segment.py
"""Render one segment. Splitting into scenes avoids long notebook/runtime timeouts."""
import sys, argparse, json
from pathlib import Path
import pandas as pd
import cv2
from make_video import FPS,W,H,scene_frame,ds_frame,SCENE_W,SCENE_H
from make_video import simulate_route,Fab3D

p=argparse.ArgumentParser();p.add_argument('scene');p.add_argument('start',type=int);p.add_argument('end',type=int);p.add_argument('--root',default=None)
a=p.parse_args()
root=Path(a.root) if a.root else Path(__file__).resolve().parents[1]
out=root/'outputs';d=out/'segments';d.mkdir(exist_ok=True)
met=json.loads((out/'metrics.json').read_text());routes=json.loads((out/'routes.json').read_text())
risk=pd.read_csv(out/'demo_ranking.csv');demo=pd.read_csv(out/'demo_equipment_logs.csv')
traces={k:simulate_route(v) for k,v in routes.items()}
s3d=Fab3D(SCENE_W,SCENE_H) if a.scene in ('intro','fixed','ai') else None
file=d/f'{a.scene}_{a.start:03d}_{a.end:03d}.mp4'
w=cv2.VideoWriter(str(file),cv2.VideoWriter_fourcc(*'mp4v'),FPS,(W,H))
if not w.isOpened():raise RuntimeError('Could not create '+str(file))
for j in range(a.start*FPS,a.end*FPS):
 t=j/FPS
 if s3d:fr=scene_frame(a.scene,t,risk,met,routes,demo,traces,s3d)
 else:fr=ds_frame(a.scene,t,risk,met,out)
 w.write(fr)
w.release()
print('ok',file,flush=True)


In [ ]:
%%writefile src/make_report.py
"""Generate a Japanese report from simulation outputs. Run fabwatch.py first."""
from pathlib import Path
import json,math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from reportlab.pdfgen import canvas
from reportlab.lib.colors import HexColor
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from reportlab.lib.utils import ImageReader
from reportlab.lib.pagesizes import A4

F='HeiseiKakuGo-W5'
pdfmetrics.registerFont(UnicodeCIDFont(F))
ROOT=Path(__file__).resolve().parents[1]
C={'navy':'#112232','blue':'#1F5F81','teal':'#008B94','ink':'#172B39','muted':'#526471','gray':'#E7EEF1','light':'#F3F7F8','orange':'#B45F22','red':'#D04E47','white':'#FFFFFF'}
PW,PH=A4

def c(hex):return HexColor(hex)

def drawtext(pdf,s,x,y,size=10,color='ink'):
    pdf.setFillColor(c(C.get(color,color)));pdf.setFont(F,size);pdf.drawString(x,y,str(s))

def wrap(pdf,s,x,y,maxw,size=10,leading=16,color='ink'):
    line=''
    for ch in s:
        if pdfmetrics.stringWidth(line+ch,F,size)>maxw:
            drawtext(pdf,line,x,y,size,color);y-=leading;line=ch
        else:line+=ch
    if line:drawtext(pdf,line,x,y,size,color);y-=leading
    return y

def rect(pdf,x,y,w,h,color):pdf.setFillColor(c(C.get(color,color)));pdf.rect(x,y,w,h,stroke=0,fill=1)

def rule(pdf,y):rect(pdf,40,y,PW-80,1,'gray')

def header(pdf,page,section):
    rect(pdf,0,PH-72,PW,72,'navy')
    drawtext(pdf,'FABWATCH',42,PH-46,24,'white')
    drawtext(pdf,section,317,PH-43,11,'#98DEE5')
    rect(pdf,42,34,PW-84,1,'gray')
    drawtext(pdf,'Physical AI 応用編1  |  模擬データ・VTKによる実3D表示',42,20,8,'muted')
    drawtext(pdf,str(page),542,20,9,'muted')

def section(pdf,n,title,y):
    rect(pdf,40,y-7,5,20,'teal')
    drawtext(pdf,f'{n}  {title}',54,y-3,13,'ink')
    return y-31

def figure(pdf,path,x,y,w,h):
    pdf.drawImage(str(path),x,y,width=w,height=h,mask='auto',preserveAspectRatio=False)


def make_charts(out):
    ev=pd.read_csv(out/'heldout_evaluation.csv')
    df=pd.read_csv(out/'demo_equipment_logs.csv')
    sched=pd.read_csv(out/'schedule_comparison.csv')
    met=json.loads((out/'metrics.json').read_text())
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
    fig,axs=plt.subplots(3,1,figsize=(7.6,4.7),sharex=True)
    for i,key in enumerate(['temp_C','vibration_mm_s','power_kW']):
        for machine,color in [('EQ04','#99A4AF'),('EQ06','#008B94')]:
            a=df[df.machine==machine]
            axs[i].plot(a.tick,a[key],color=color,lw=1.8,label=machine)
        axs[i].set_ylabel({'temp_C':'Temp (C)','vibration_mm_s':'Vib. (mm/s)','power_kW':'Power (kW)'}[key])
        axs[i].grid(alpha=.18)
    axs[0].legend(ncol=2,frameon=False,loc='upper left')
    axs[-1].set_xlabel('Synthetic time tick')
    fig.tight_layout();fig.savefig(out/'sensor_signals.png',dpi=160,facecolor='white');plt.close(fig)

    fig,ax=plt.subplots(figsize=(7.8,3.8))
    norm=ev.loc[ev.injected_fault==0,'anomaly_score'];anom=ev.loc[ev.injected_fault==1,'anomaly_score']
    bins=np.linspace(min(ev.anomaly_score),max(ev.anomaly_score),29)
    ax.hist(norm,bins=bins,alpha=.85,color='#7797A9',label=f'Healthy windows (n={len(norm)})')
    ax.hist(anom,bins=bins,alpha=.72,color='#D76F55',label=f'Injected anomalies (n={len(anom)})')
    ax.axvline(met['threshold'],color='#1C3647',lw=2,ls='--',label='Calibration threshold')
    ax.set_xlabel('Isolation Forest window anomaly score')
    ax.set_ylabel('Equipment windows')
    ax.grid(axis='y',alpha=.12)
    ax.legend(frameon=False,fontsize=9)
    fig.tight_layout();fig.savefig(out/'score_distribution.png',dpi=160,facecolor='white');plt.close(fig)

    fig,axs=plt.subplots(1,2,figsize=(8,3.15))
    names=['Fixed','AI score']
    axs[0].bar(names,[met['eval_avg_fixed_first_fault_s'],met['eval_avg_ai_first_fault_s']],color=['#9AADB8','#008B94'])
    axs[0].set_ylabel('First fault inspected (sim s)')
    axs[1].bar(names,[met['eval_avg_fixed_route_m'],met['eval_avg_ai_route_m']],color=['#9AADB8','#B78A43'])
    axs[1].set_ylabel('Total travel length (m)')
    for ax in axs:ax.grid(axis='y',alpha=.12);ax.set_axisbelow(True)
    fig.tight_layout();fig.savefig(out/'schedule_tradeoff.png',dpi=160,facecolor='white');plt.close(fig)
    return met


def build_pdf(out=None,account_id='ACCOUNT_NAME'):
    out=Path(out or (ROOT/'outputs'));out.mkdir(parents=True,exist_ok=True)
    met=make_charts(out)
    report=out/f'説明資料_PAI最終課題_{account_id}.pdf'
    pdf=canvas.Canvas(str(report),pagesize=A4)
    pdf.setTitle('FabWatch: 模擬設備の異常検知と3D点検ロボット')
    pdf.setAuthor('anonymous')
    # PAGE 1: submission rubric explicitly asks for objective and ingenuity.
    header(pdf,1,'目的・課題概要・工夫した点')
    y=PH-103
    drawtext(pdf,'FabWatch：AI異常検知を使う3D点検ロボット',42,y,17)
    y-=24
    drawtext(pdf,'Synthetic equipment logs / Inspection robot',42,y,10,'teal')
    y-=28
    y=section(pdf,'01','目的',y)
    y=wrap(pdf,'模擬製造設備の温度・振動・消費電力から異常度を推定し、異常度に応じてロボットの点検順序を変更する仕組みを実装する。固定順巡回と比較し、初回異常設備の点検までの時間と移動距離がどう変わるかを調べる。',42,y,PW-84,9.6,15)
    y-=9
    y=section(pdf,'02','課題概要と処理の流れ',y)
    steps=[
        ('模擬ログの作成','8設備の時系列データを人工生成し、SQLiteへ保存'),
        ('AIによる異常検知','正常ログでIsolation Forestを学習し、設備別スコアを推定'),
        ('巡回順序の決定','設備番号順と異常スコア優先順の2方式を比較'),
        ('3Dロボットによる点検','A*の経路と差動二輪の簡略運動モデルで走行・停止を表示')]
    for i,(head,detail) in enumerate(steps):
        rect(pdf,43,y-30,507,35,'light')
        drawtext(pdf,f'{i+1:02d}',54,y-9,13,'teal')
        drawtext(pdf,head,89,y-5,9.5)
        drawtext(pdf,detail,89,y-20,8.4,'muted')
        y-=44
    y-=4
    y=section(pdf,'03','工夫した点',y)
    details=[
        ('学習と評価を分離','正常データで学習し、別の正常データで閾値を設定。未学習の60条件で検証した。'),
        ('AIの判断をロボットの動作へ接続','異常スコアを巡回順位に反映し、A*探索で訪問経路を求めた。'),
        ('利点だけでなく制約も評価','初回点検時間に加え、増加した移動距離も同じ条件で比較した。')]
    for head,detail in details:
        drawtext(pdf,'・'+head,43,y,9.5,'ink');y-=15
        drawtext(pdf,detail,54,y,8.25,'muted');y-=22
    # Regenerate PDF after generating 3D preview so this image is not lost.
    if (out/'3d_inspection.png').exists():
        figure(pdf,out/'3d_inspection.png',122,44,351,197)
    else:
        rect(pdf,122,44,351,197,'light')
        drawtext(pdf,'実3D映像のプレビュー画像は動画生成後に表示されます。',139,134,9,'muted')
    pdf.showPage()

    # PAGE 2
    header(pdf,2,'異常検知モデルと評価')
    y=PH-103
    y=section(pdf,'04','学習データと評価条件',y)
    y=wrap(pdf,'設備ごとの基準値と周期変動にノイズを加え、正常ログを生成した。異常条件には温度・振動・電力が継続的に増加するドリフトを注入した。正常データだけでIsolation Forestを学習し、別に生成した正常ログのスコア分布95パーセンタイルで閾値を設定した。',42,y,PW-85,9.5,15)
    y-=13
    figure(pdf,out/'score_distribution.png',49,y-171,491,169)
    y-=185
    y=section(pdf,'05','未学習60条件での検証',y)
    rect(pdf,42,y-47,510,57,'light')
    drawtext(pdf,f"Precision {met['precision']:.3f}     Recall {met['recall']:.3f}     F1 {met['f1']:.3f}",52,y-8,12,'teal')
    drawtext(pdf,f"TP={met['tp']} / FP={met['fp']} / FN={met['fn']} / TN={met['tn']}  （設備窓480件）",52,y-31,9.5)
    y-=70
    y=wrap(pdf,'異常の有無はデータ生成時の注入条件から定義した。モデルにはそのラベルを学習させていない。閾値選定にも異常ラベルを使用していない。評価の60条件は学習・閾値設定と異なる乱数系列で生成した。',42,y,PW-85,9.5,15)
    y-=11
    rect(pdf,42,y-63,510,70,'#FFF3E6')
    drawtext(pdf,'結果の解釈',52,y-12,10,'orange')
    wrap(pdf,'異常は各センサー値が一定期間増えるように人工的に設定した。実際の設備故障を予測できるかどうかは、この結果からは判断できない。',52,y-29,490,9,14)
    pdf.showPage()

    # PAGE 3
    header(pdf,3,'ロボットの巡回比較')
    y=PH-103
    y=section(pdf,'06','巡回方針の比較',y)
    y=wrap(pdf,'固定方式はEQ01からEQ08を順番に点検する。AI方式はIsolation Forestの異常度が高い順に点検する。どちらも同じ地図・A*探索・走行速度0.74m/s・1台あたり点検6秒という条件で計算した。動画では差動二輪の簡略運動モデルが各設備で停止し、点検中の設備名を表示する。',42,y,PW-85,9.5,15)
    y-=12
    figure(pdf,out/'schedule_tradeoff.png',53,y-189,485,187)
    y-=204
    y=section(pdf,'07','定量結果',y)
    rows=[('初回異常点検までの平均時間',f"{met['eval_avg_fixed_first_fault_s']:.1f} 秒",f"{met['eval_avg_ai_first_fault_s']:.1f} 秒"),
          ('平均移動距離',f"{met['eval_avg_fixed_route_m']:.1f} m",f"{met['eval_avg_ai_route_m']:.1f} m"),
          ('AI方式で点検が早い条件','―','42 / 60 条件')]
    rect(pdf,42,y-18,509,25,'navy')
    drawtext(pdf,'比較項目',53,y-10,9,'white');drawtext(pdf,'固定順',324,y-10,9,'white');drawtext(pdf,'AI優先',432,y-10,9,'white')
    y-=21
    for j,(a,b,d) in enumerate(rows):
        rect(pdf,42,y-27,509,30,'light' if j%2==0 else 'white')
        drawtext(pdf,a,52,y-16,9);drawtext(pdf,b,319,y-16,9);drawtext(pdf,d,417,y-16,8.3)
        y-=30
    y-=13
    y=section(pdf,'08','制約と改善点',y)
    y=wrap(pdf,'AI方式は60条件中42条件で初回異常点検が早く、14条件で遅く、4条件で同じだった。一方、平均移動距離は長くなった。異常スコアだけで訪問順を決めるため、移動距離を考慮できていない。',42,y,PW-85,9.5,15)
    y-=12
    y=wrap(pdf,'今後は異常スコアと移動時間を併せて点検順を決めたい。定量結果はA*の計画経路と一定速度の仮定に基づく。VTKで実3D表示を行ったが、剛体運動、タイヤ摩擦、衝突応答を物理エンジンで計算したものではない。距離センサーは設備への簡略的なレイ交差で表現した。',42,y,PW-85,9.2,15)
    pdf.showPage();pdf.save()
    return report

if __name__=='__main__':print(build_pdf())


In [ ]:
%%writefile src/build_submission.py
"""Rebuild metrics, charts, Japanese PDF and 3D movie. Supports resuming completed video segments."""
import sys, subprocess, os, cv2, math
from pathlib import Path
from make_video import add_op_ed, FPS
from make_report import build_pdf, make_charts
ROOT=Path(__file__).resolve().parents[1]
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
segments=[('intro',0,6),('train',0,13),('validate',0,11),
          ('fixed',0,16),('fixed',16,32),('fixed',32,48),
          ('ai',0,14),('ai',14,28),('ai',28,42),('ai',42,56),
          ('results',0,13),('limitations',0,6)]

def run(cmd):
    print('RUN', ' '.join(str(x) for x in cmd), flush=True)
    subprocess.run([str(a) for a in cmd],check=True,cwd=ROOT)

def valid_segment(path, seconds):
    """An interrupted MP4 may exist but have no playable frames."""
    if not path.exists() or path.stat().st_size < 10000:
        return False
    cap=cv2.VideoCapture(str(path))
    ok=cap.isOpened()
    fps=cap.get(cv2.CAP_PROP_FPS) if ok else 0
    frames=cap.get(cv2.CAP_PROP_FRAME_COUNT) if ok else 0
    cap.release()
    return bool(ok and fps>0 and frames>=seconds*fps-2)

if __name__=='__main__':
    # Recalculate model and numerical metrics from the same fixed-seed source.
    run([sys.executable,ROOT/'src/fabwatch.py'])
    # Important: charts MUST be created before train/validate video segments.
    make_charts(OUT)
    pdf=build_pdf(OUT,account_id=os.environ.get('OMNI_ACCOUNT_ID','ACCOUNT_NAME'))
    print('PDF prepared:',pdf, flush=True)
    for scene,a,b in segments:
        path=OUT/'segments'/f'{scene}_{a:03d}_{b:03d}.mp4'
        if valid_segment(path, b-a):
            print('SKIP valid segment:',path.name,flush=True)
            continue
        if path.exists():path.unlink()
        run([sys.executable,ROOT/'src/render_segment.py',scene,a,b])
        if not valid_segment(path,b-a):
            raise RuntimeError('Rendering did not produce a valid segment: '+str(path))
    ff_list=OUT/'segments'/'concat_list.txt'
    ff_list.write_text(''.join("file '"+str(OUT/'segments'/f'{sc}_{a:03d}_{b:03d}.mp4')+"'\n" for sc,a,b in segments),encoding='utf-8')
    movie=OUT/'FabWatch_実3D_本編.mp4'
    run(['ffmpeg','-hide_banner','-loglevel','error','-y','-f','concat','-safe','0','-i',ff_list,'-c:v','libx264','-preset','veryfast','-crf','22','-pix_fmt','yuv420p',movie])
    cap=cv2.VideoCapture(str(movie));cap.set(cv2.CAP_PROP_POS_MSEC,94*1000)
    ok,frame=cap.read();cap.release()
    if ok:cv2.imwrite(str(OUT/'3d_inspection.png'),frame)
    # Rebuild report after screenshot extraction to include the true 3D preview.
    pdf=build_pdf(OUT,account_id=os.environ.get('OMNI_ACCOUNT_ID','ACCOUNT_NAME'))
    print('Final PDF with 3D preview:',pdf,flush=True)
    op,ed=ROOT/'assets/OP.mp4',ROOT/'assets/ED.mp4'
    if op.exists() and ed.exists():
        print('FINAL',add_op_ed(ROOT,movie,op,ed),flush=True)
    else:
        print('OP/ED not present. Main:',movie,flush=True)


## 4. 実行（修正版：グラフ作成後に動画生成）

モデル・評価結果を計算してから、グラフ画像とPDFを作成し、3D動画を生成します。
中断した場合は同じセルを再実行してください。再生可能な動画パートは自動で再利用します。
`OMNI_ACCOUNT_ID` はPDFの指定ファイル名のため実行時に自分のOmniCampus IDに変更してください。


In [ ]:
OMNI_ACCOUNT_ID='ACCOUNT_NAME'  # 実行時のみアカウント名を入力。公開用コードには保存しない
os.environ['OMNI_ACCOUNT_ID']=OMNI_ACCOUNT_ID
!python src/build_submission.py

## 5. 結果の確認
合成異常の検出性能と、計画経路に基づく固定巡回／AI優先巡回を確認する。

In [ ]:
import json
import pandas as pd
from IPython.display import display,Video
m=json.loads((ROOT/'outputs'/'metrics.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'Precision':m['precision'],'Recall':m['recall'],'F1':m['f1'],'固定方式_平均初回点検秒':m['eval_avg_fixed_first_fault_s'],'AI方式_平均初回点検秒':m['eval_avg_ai_first_fault_s']}]))
main=ROOT/'outputs'/'FabWatch_実3D_本編.mp4'
print('本編',main)
display(Video(str(main),embed=True,width=780))

## 6. OP・EDの挿入（使用許諾がある場合のみ）
講義から提供されたOP/ED動画は自動で公開リポジトリに含めない。希望する場合は `assets/OP.mp4` と `assets/ED.mp4` を手動で配置する。

In [ ]:
from pathlib import Path
from google.colab import files
import sys
import subprocess
import re

# 作業フォルダ
ROOT = Path("/content/FabWatch")
assets = ROOT / "assets"
assets.mkdir(parents=True, exist_ok=True)

# ① OP・EDのアップロード
print("OPとEDの2本のMP4ファイルを選択してください")
uploaded = files.upload()

# 重複アップロードによる「 (1).mp4」などのサフィックスを補正するマッピング処理
cleaned_uploaded = {}
for filename, content in uploaded.items():
    # 例: "PAI応用1_最終課題_OP (4).mp4" -> "PAI応用1_最終課題_OP.mp4"
    cleaned_name = re.sub(r"\s*\(\d+\)\.mp4$", ".mp4", filename)
    cleaned_uploaded[cleaned_name] = content

mapping = {
    "PAI応用1_最終課題_OP.mp4": "OP.mp4",
    "PAI応用1_最終課題_ED.mp4": "ED.mp4"
}

for original, renamed in mapping.items():
    if original not in cleaned_uploaded:
        raise FileNotFoundError(f"アップロードされていません: {original} (アップロードしたファイル名を確認してください)")
    (assets / renamed).write_bytes(cleaned_uploaded[original])

# ② 本編動画を確認
main = ROOT / "outputs" / "FabWatch_実3D_本編.mp4"

if not main.exists():
    raise FileNotFoundError("本編動画がありません")

# ③ OP + 本編 + ED を結合
sys.path.insert(0, str(ROOT / "src"))
from make_video import add_op_ed

final = add_op_ed(
    ROOT,
    main,
    assets / "OP.mp4",
    assets / "ED.mp4"
)

# ④ 動画の長さを確認
duration = float(subprocess.check_output(
    [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(final)
    ],
    text=True
).strip())

print(f"完成動画: {final}")
print(f"再生時間: {duration:.1f} 秒")

if duration > 180:
    raise ValueError("3分を超えているため確認が必要です")

# ⑤ 完成動画をダウンロード
files.download(str(final))

## 7. 提出物を保存
PDF・MP4・CSVの値と公開設定を確認する。

In [ ]:
import shutil
archive=shutil.make_archive(str(ROOT/'FabWatch_outputs'),'zip',ROOT/'outputs')
print('ZIP:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass